In [1]:
# Load the ground truth questions:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
# Load the FAQ documents and the search index:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
#Create a lookup table for the original FAQ documents 
# to find the original answer for each ground truth question:

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
doc_idx['74eb249bbf']

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [5]:
# Running Rag
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
from evaluation_utils import RAGWithUsage
# discovered values are from search_evaluation: "question": 2.0, "answer": 10.0, "section": 0.2,

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    boost_dict={"question": 2.0, "answer": 10.0, "section": 0.2},
)

In [7]:
#Run RAG for one question:

rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
print(f"question: {question}")
print(f"answer: {answer_llm}")

question: I just found this course — am I still allowed to join, or is it too late?
answer: Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.


In [8]:
# check the total cost:
assistant.total_cost()

0.0007559999999999999

In [9]:
# Get the original answer by document id:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [10]:
# Now save both answers in one record:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course — am I still allowed to join, or is it too late?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [11]:
# Create a function that processes one ground truth record:

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [12]:
# Test on one record
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course — am I still allowed to join, or is it too late?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [13]:
# reset the usage before running this on the full set of records:
assistant.reset_usage()

In [14]:
# Import the parallel processing helper from the same utility file:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [15]:
# Run RAG for all ground truth questions:
with ThreadPoolExecutor(max_workers=9) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/565 [00:00<?, ?it/s]

In [16]:
# Collect the answer records:
answers = []

for answer_record in results:
    answers.append(answer_record)


In [17]:
# Calculate total cost:
assistant.total_cost()


0.6152992500000004

In [18]:
# save the answers:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [19]:
## LLM as judge
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [20]:
# A->Q->A' evaluation
# First, define the output format
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [ ]:
# The judge returns two fields: score and reasoning
#  The score gives us a metric we can aggregate. 
# The reasoning explains the score, which helps when we look at bad examples.

In [21]:
# Write judge instructions:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [22]:
# Define prompt template:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [23]:
# Import structured output helper:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [25]:
# Take one record:
rec = answers[0]; rec

{'question': 'I just found this course — am I still allowed to join, or is it too late?',
 'answer_llm': 'Yes, you can still join. It’s not too late.\n\nIf you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [26]:
## Create the judge prompt:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [27]:
# Call the judge:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says the course can still be joined and correctly adds that certificate eligibility requires submitting the project while submissions are still being accepted. Semantically equivalent.', score='good')

In [28]:
# Check the cost:
calc_price(usage)

{'input_cost': 0.00022650000000000003,
 'output_cost': 0.0002385,
 'total_cost': 0.000465}

In [30]:
# Put the same logic into the function:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage


In [31]:
# Test it on the same record:

eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: joining is still allowed, but certificate eligibility depends on submitting the project while submissions are still open. This is semantically equivalent.', score='good')

In [32]:
# Run the judge on all answers:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [33]:
# USe the same parallel helper:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/565 [00:00<?, ?it/s]

In [34]:
# Split the results:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [35]:
# Create dataframe:
df_eval = pd.DataFrame(evaluations)

In [36]:
# Calculate the total cost:
calc_total_price(usages)

0.40082624999999994

In [37]:
# Check the results:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 542/565 = 95.93%


In [38]:
# Look at the bad results to see what went wrong:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
4,Can I enroll after the course has already star...,74eb249bbf,bad,The AI answer does not preserve the key ground...
31,Is the Capstone project enough to get the cour...,9f689c185f,bad,The AI answer includes the key point that pass...
32,Are homework assignments required for certific...,9f689c185f,bad,The AI answer contradicts the ground truth. Th...
40,When is the llm-zoomcamp going to be offered a...,bd31146b0e,bad,The ground truth specifies a concrete timefram...
41,Do you know the next time this course will run?,bd31146b0e,bad,The ground truth gives a specific next run tim...


In [39]:
# These rows are often the most useful part of the evaluation. 
# They can show that search retrieved the wrong document. 
# They can also show that the answer is too generic. 
# Sometimes the RAG pipeline says that it doesn't know even though the FAQ had the answer.

In [43]:
# Save the results:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)